In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

assignment_2_speech_translation_path = kagglehub.competition_download('assignment-2-speech-translation')

print('Data source import complete.')


Data source import complete.


# Vi → En Speech Translation - Starter Notebook

A **real, runnable** Vietnamese→English speech-translation pipeline: it loads an allowed Whisper model, fine-tunes it with LoRA on a public dataset, translates all **2,481** competition test clips, and writes `submission.csv`.

It is deliberately a **starting point, not a finish line.** The defaults here are plain, not tuned - running it as-is gets you on the leaderboard with a *modest* score. There is a lot of room above it, and you only reach that room by **reading the code, understanding what each step does, and experimenting.** Hitting "Run All" is not a strategy.

> Read every cell before you trust it. The pipeline runs and the loss goes down - that does **not** mean every line is doing the right thing. The final section tells you how to find out.

**Before you run:** Settings → Accelerator → **GPU T4 ×2** (or any GPU). Settings → Internet → **On** (the first run downloads the model + dataset; both are cached afterwards).

**Rules reminder.** Only Whisper variants are allowed. Any public ASR/MT dataset is allowed. No paid inference APIs. **5 submissions per day** - so measure your ideas locally before spending one (see the final section).

### Step 0 - set up the environment

Nothing in this step affects your score; it just gets Kaggle ready, so you can **skim it**. First we confirm a GPU is attached, then we install a few extra packages Kaggle doesn't ship by default.

### Mount Google Drive
This step will mount your Google Drive, allowing the notebook to save checkpoints directly to your Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Next, we will modify the `Seq2SeqTrainingArguments` to save all checkpoints to a specified folder in your Google Drive. I will use `'/content/drive/MyDrive/whisper-medium-checkpoints'` as the output directory. You can change this path if you wish.

In [ ]:
import torch

# Confirm a GPU is attached - training and inference both need one.
assert torch.cuda.is_available(), \
    "No GPU found. Settings → Accelerator → GPU T4 ×2 (or any GPU)."

# Quick sanity check that CUDA actually works (catches a broken runtime early).
_ = torch.randn(8, 8, device="cuda") @ torch.randn(8, 8, device="cuda")
print(f"torch={torch.__version__} | {torch.cuda.get_device_name(0)}")

torch=2.11.0+cu128 | Tesla T4


**Install the extra packages.** Kaggle already includes PyTorch and Transformers - we just add the few that are missing. `--no-deps` tells pip *not* to reinstall things Kaggle already set up (it stops pip from quietly swapping in a different PyTorch and breaking the GPU).

In [ ]:
import importlib, subprocess, sys

# datasets<3.0 - the FLEURS loader script was removed in datasets 3.x, so we pin an older version.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "datasets<3.0"])

# Install or upgrade torchao to a compatible version before peft.
# peft requires torchao >= 0.16.0.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torchao>=0.16.0"])

# peft = the LoRA library. --no-deps so pip doesn't try to swap out Kaggle's PyTorch.
if importlib.util.find_spec("peft") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "peft"])

# evaluate + sacrebleu - used to measure BLEU.
for pkg in ("evaluate", "sacrebleu"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# Re-check CUDA still works after installing (a bad install can break torch).
_ = torch.randn(8, 8, device="cuda") @ torch.randn(8, 8, device="cuda")
print("CUDA OK")

CUDA OK


In [ ]:
import sys
import subprocess

# Install evaluate and sacrebleu if not already present
# This is added here to address the ModuleNotFoundError for 'evaluate'
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "evaluate", "sacrebleu"])

# Standard library + scientific Python
import gc, os
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Union

import numpy as np
import pandas as pd
import soundfile as sf          # reads .wav audio files
import torch

# Hugging Face: datasets, LoRA (peft), and the Whisper model + Trainer
import evaluate
from datasets import Audio, DatasetDict, load_dataset, load_from_disk
from peft import LoraConfig, get_peft_model
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    WhisperForConditionalGeneration,
    WhisperProcessor,
)

**The knobs.** These are the settings you'll spend the competition experimenting with. They're at plain, modest defaults on purpose - changing them (and understanding *why* a change helps) is the whole point.

In [ ]:
# --- The knobs. Plain starting points, not tuned values - these are yours to change. ---
MODEL_ID         = "openai/whisper-medium"   # an allowed Whisper variant
NUM_EPOCHS       = 2                        # how many passes over the training data
TRAIN_BATCH      = 8                       # clips per training step (per GPU)
GRAD_ACCUM       = 4                         # effective batch = TRAIN_BATCH * GRAD_ACCUM
LEARNING_RATE    = 1e-3                       # how big each training step is
INFER_BATCH      = 8                         # clips translated at once at the end
INFER_BEAMS      = 8                         # beams used when decoding
SAMPLING_RATE    = 16_000                    # Whisper expects 16 kHz audio

**Where things get saved.** Paths for the cached dataset, the fine-tuned model, and the `submission.csv` that Kaggle scores.

In [ ]:
# Where files get read from / written to. On Kaggle this is /kaggle/working.
WORK_DIR     = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
FLEURS_CACHE = WORK_DIR / "fleurs_vi_en"        # cached copy of the training data
OUT_DIR      = WORK_DIR / "whisper-large-ft"    # where the fine-tuned adapter is saved
SUBMISSION   = WORK_DIR / "submission.csv"      # the file Kaggle scores

device = "cuda"
print(f"working dir: {WORK_DIR}")

working dir: /content


## 1. Locate the competition test set

The competition dataset is mounted under `/kaggle/input/`. The block below grabs every `test/*.wav` in your attached competition data automatically. Running locally? Make sure the test wavs live in some `<dir>/test/*.wav` structure.

In [ ]:
def find_test_dir() -> Path:
    """Find the folder of competition test .wav files under /kaggle/input/.

    The dataset can be attached inside a versioned subfolder, so we look at every
    `*/test` directory and keep whichever one actually holds the most .wav files.
    """
    # Update comp_root to use the path where kagglehub downloaded the data.
    # The variable assignment_2_speech_translation_path holds this path.
    comp_root = Path(assignment_2_speech_translation_path)

    candidates = []
    if comp_root.exists():
        candidates = sorted(comp_root.glob("*/test"))

    if not candidates:
        raise FileNotFoundError(
            "No test/ directory found under the specified competition data path. "
            "Ensure the dataset was downloaded correctly and check its internal structure."
        )

    # Pick the candidate folder containing the most .wav files.
    return max(candidates, key=lambda p: len(list(p.glob("*.wav"))))


TEST_DIR  = find_test_dir()
TEST_WAVS = sorted(TEST_DIR.glob("*.wav"))   # every test clip, sorted by filename
print(f"Test dir : {TEST_DIR}")
print(f"Test wavs: {len(TEST_WAVS)}")

Test dir : /root/.cache/kagglehub/competitions/assignment-2-speech-translation/public_release_v2/test
Test wavs: 2481


## 2. Load FLEURS Vi/En as training data

FLEURS pairs Vietnamese audio with English sentence-aligned transcriptions. ~2.8k Vi training clips, ~5 s each - small but enough for a meaningful LoRA fine-tune.

**Want to swap in a different corpus?** Replace this cell with whatever public Vi-audio + En-text dataset you like (CoVoST 2, VIVOS + MT'd labels, CommonVoice + MT, etc.). The rest of the pipeline is dataset-agnostic as long as `dataset["train"]` and `dataset["validation"]` end up with `audio` and `translation` columns.

In [ ]:
# FLEURS ships Vietnamese and English separately; we pair them by their shared `id`
# to build (Vietnamese audio, English text) examples. The result is cached to disk so
if FLEURS_CACHE.is_dir():
    dataset = load_from_disk(str(FLEURS_CACHE))
    print(f"Loaded FLEURS from cache: {FLEURS_CACHE}")
else:
    print("Downloading FLEURS vi_vn + en_us (one-time, ~5 min)…")
    fleurs_vi = load_dataset("google/fleurs", "vi_vn", trust_remote_code=True)
    fleurs_en = load_dataset("google/fleurs", "en_us", trust_remote_code=True)

    splits = {}
    for split in ("train", "validation", "test"):
        # Map each example id -> its English sentence.
        id_to_en = {ex["id"]: ex["transcription"] for ex in fleurs_en[split]}

        # Keep only Vietnamese clips that have a matching English sentence,
        # then attach that sentence as a new `translation` column.
        vi = fleurs_vi[split].filter(lambda ex: ex["id"] in id_to_en)
        vi = vi.map(lambda ex: {"translation": id_to_en[ex["id"]]})

        # Drop every column except the audio and its English translation.
        keep = ("audio", "translation")
        vi = vi.remove_columns([c for c in vi.column_names if c not in keep])
        splits[split] = vi

    # Tell the dataset the audio is 16 kHz, then save the paired data to disk.
    dataset = DatasetDict(splits).cast_column("audio", Audio(sampling_rate=SAMPLING_RATE))
    dataset.save_to_disk(str(FLEURS_CACHE))

print({k: len(v) for k, v in dataset.items()})
print("Sample translation:", dataset["train"][0]["translation"])

Loaded FLEURS from cache: /content/fleurs_vi_en
{'train': 2927, 'validation': 361, 'test': 857}
Sample translation: following the process hjr-3 will be reviewed again by the next elected legislature in either 2015 or 2016 to remain in process


## 3. Prepare the data: processor, collator, and metric

Three small pieces the Trainer needs: the **processor** (converts audio/text ↔ numbers), the **data collator** (packs examples into training batches), and the **BLEU metric** (scores the validation set so you can watch it learn).

In [ ]:
# The processor bundles two tools: a feature extractor (audio → log-mel spectrogram,
# the image-like input Whisper actually reads) and a tokenizer (text ↔ token ids).
# language="vi", task="translate" sets the job: Vietnamese audio in, English text out.
processor = WhisperProcessor.from_pretrained(MODEL_ID, language="vi", task="translate")

**The data collator.** A "collator" is the function that turns a handful of raw examples into one padded batch of tensors the model can train on. This one works in four steps: audio → features, text → token ids, mask the padding, drop the leading start token.

In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    """Turns a list of {audio, translation} examples into one padded batch for training."""
    processor: Any

    def __call__(self, features):
        # 1) Audio → log-mel features (Whisper's actual input).
        arrays = [np.array(f["audio"]["array"], dtype=np.float32) for f in features]
        batch = self.processor.feature_extractor(
            arrays, sampling_rate=SAMPLING_RATE, return_tensors="pt",
            padding="max_length", truncation=True,
            # padding=True,
        )
        # The model runs in bf16; generate() isn't autocast-wrapped, so match the dtype here.
        batch["input_features"] = batch["input_features"].to(torch.bfloat16)

        # 2) English text → token ids (the labels the model learns to produce).
        label_features = [
            {"input_ids": self.processor.tokenizer(f["translation"], truncation=True, max_length=225).input_ids}
            for f in features
        ]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # 3) Set padded label positions to -100 so the loss ignores them.
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # 4) Drop the leading start-of-sentence token - the model re-adds it internally.
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

**Validation metric - BLEU.** During training the Trainer translates the validation clips and scores them with BLEU (higher = closer to the reference English), so you can see learning happen instead of guessing.

In [ ]:
# BLEU measures how close the model's English output is to the reference translation.
# The Trainer calls this each epoch on the validation split so you can watch it improve.
bleu_metric = evaluate.load("sacrebleu")

In [ ]:
def compute_metrics(pred):
    pred_ids, label_ids = pred.predictions, pred.label_ids
    # Undo the -100 masking so the labels can be decoded back into text.
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    bleu = bleu_metric.compute(predictions=pred_str, references=[[l] for l in label_str])
    return {"bleu": bleu["score"]}

## 4. Build the model

We load the base Whisper model, then attach small **LoRA** adapters so only a tiny fraction of the parameters actually train. That's what lets fine-tuning fit on a free T4 GPU.

In [ ]:
# Load the base Whisper model in bf16 (half precision → fits a T4, trains faster).
base = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, attn_implementation="sdpa"
)

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

In [ ]:
# Tell generation: Vietnamese audio → translate to English.
base.generation_config.language = "vi"
base.generation_config.task = "translate"

# Clear stale generation defaults so the settings we pass at predict time are the only ones in effect.
base.generation_config.num_beams = 1
base.generation_config.no_repeat_ngram_size = 0
base.generation_config.suppress_tokens = []
base.generation_config.forced_decoder_ids = None
base.config.suppress_tokens = None
base.config.begin_suppress_tokens = None
base.config.forced_decoder_ids = None

**Attach LoRA adapters.** Instead of retraining all ~240M parameters, we freeze them and train a small set of add-on matrices. That's what makes fine-tuning feasible on a free GPU.

In [ ]:
# LoRA = freeze the big pretrained model and train only tiny "adapter" matrices added
# into each linear layer. Far fewer trainable parameters → fits in T4 memory, trains fast.
# r and lora_alpha control the adapter size; they're starting points, not tuned values.
lora_cfg = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"],
    bias="none",
)

In [ ]:
model = get_peft_model(base, lora_cfg)
model.enable_input_require_grads()
model.print_trainable_parameters()   # prints how few parameters actually train

trainable params: 14,417,920 || all params: 1,557,908,480 || trainable%: 0.9255


## 5. Train

First the training settings (grouped by purpose), then we build the Trainer and run the loop. `load_best_model_at_end=True` keeps the checkpoint with the highest validation BLEU, not just the final one.

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir='/content/drive/MyDrive/whisper-large-checkpoints',

    # --- how long / how big ---
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,

    # --- learning schedule ---
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,

    # --- memory / precision ---
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=False, bf16=True,
    optim="adamw_torch",

    # --- evaluation & checkpoints (score BLEU + save once per epoch, keep the best) ---
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_num_beams=1,
    save_total_limit=None, # Changed to None to save all checkpoints
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    greater_is_better=True,

    # --- misc ---
    report_to=[],
    push_to_hub=False,
    dataloader_num_workers=2,
    remove_unused_columns=False,
    label_names=["labels"],
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


## Validation

In [ ]:
def translate_audio_examples(model, audio_examples, num_beams):
    model.eval()

    forced_ids = processor.get_decoder_prompt_ids(
        language="vi",
        task="translate"
    )

    outputs = []

    for start in range(0, len(audio_examples), INFER_BATCH):

        batch = audio_examples[start:start+INFER_BATCH]

        arrays = [
            np.array(x["array"], dtype=np.float32)
            for x in batch
        ]

        inputs = processor.feature_extractor(
            arrays,
            sampling_rate=SAMPLING_RATE,
            return_tensors="pt",
            padding=True
        )

        feats = inputs.input_features.to(
            device,
            dtype=next(model.parameters()).dtype
        )

        with torch.no_grad():

            ids = model.generate(
                feats,
                forced_decoder_ids=forced_ids,
                num_beams=num_beams,
                no_repeat_ngram_size=2,
            )

        outputs.extend(
            processor.tokenizer.batch_decode(
                ids,
                skip_special_tokens=True
            )
        )

    return outputs

**Build the trainer and run.** `Seq2SeqTrainer` wires together the model, data, collator, and metric; `train()` runs the loop and prints loss + validation BLEU as it goes.

In [ ]:
import sacrebleu
from transformers.trainer_utils import get_last_checkpoint

val_wavs = []
val_refs = []

for ex in dataset["validation"]:
    val_wavs.append(ex["audio"])
    val_refs.append(ex["translation"])

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=DataCollatorSpeechSeq2SeqWithPadding(processor=processor),
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

# # Ensure the output directory exists before trying to access it for checkpoints
# OUT_DIR.mkdir(parents=True, exist_ok=True)
# checkpoint = get_last_checkpoint(str(OUT_DIR))

# trainer.train(resume_from_checkpoint=checkpoint)

# ft_model = model.merge_and_unload().to(device)

# for beams in [8]:

#     preds = translate_audio_examples(
#         ft_model,
#         val_wavs,
#         num_beams=beams
#     )

#     bleu = sacrebleu.corpus_bleu(
#         preds,
#         [val_refs]
#     )

#     print(
#         f"Beam={beams}  BLEU={bleu.score:.2f}"
#     )

# # Save the trained LoRA adapter + processor so the prediction step can load them.
# model.save_pretrained(str(OUT_DIR))
# processor.save_pretrained(str(OUT_DIR))
# print(f"LoRA adapter saved to {OUT_DIR}")

NameError: name 'model' is not defined

In [ ]:
del trainer
torch.cuda.empty_cache()

In [ ]:
from peft import PeftModel, PeftConfig # Import PeftConfig
from transformers import WhisperForConditionalGeneration, WhisperProcessor
import torch
import os # Import os for path manipulation

CHECKPOINT_PATH = "/content/drive/MyDrive/checkpoint_best"  # your checkpoint

MODEL_ID = "openai/whisper-medium"  # Reverted to match user's clarification

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = WhisperProcessor.from_pretrained(
    MODEL_ID,
    language="vi",
    task="translate"
)

base_model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
)

# --- Robust Local PEFT Model Loading (bypassing HF Hub validation) ---
# 1. Load the PeftConfig from the local checkpoint path
peft_config = PeftConfig.from_pretrained(CHECKPOINT_PATH)

# 2. Create the PeftModel from the base model and the config
# This initializes the PEFT layers without loading weights yet.
model = PeftModel(base_model, peft_config)

# 3. Load the adapter weights directly into the PeftModel
# This assumes the adapter weights are in adapter_model.bin or adapter_model.safetensors
adapter_weights_path_bin = os.path.join(CHECKPOINT_PATH, "adapter_model.bin")
adapter_weights_path_safetensors = os.path.join(CHECKPOINT_PATH, "adapter_model.safetensors")

if os.path.exists(adapter_weights_path_bin):
    adapter_weights = torch.load(adapter_weights_path_bin, map_location="cpu")
elif os.path.exists(adapter_weights_path_safetensors):
    from safetensors import safe_open
    adapter_weights = {}
    with safe_open(adapter_weights_path_safetensors, framework="pt", device="cpu") as f:
        for key in f.keys():
            adapter_weights[key] = f.get_tensor(key)
else:
    raise FileNotFoundError(
        f"Adapter weights file (adapter_model.bin or adapter_model.safetensors) not found in {CHECKPOINT_PATH}"
    )

# Load the state dictionary into the PeftModel
model.load_state_dict(adapter_weights, strict=False) # strict=False to ignore keys not in the adapter_weights
# -----------------------------------------------------------------------

model = model.merge_and_unload().to(device)

model.eval()

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 1024, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(1024, 1024, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 1024)
      (layers): ModuleList(
        (0-23): 24 x WhisperEncoderLayer(
          (self_attn): WhisperAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=Tru

In [ ]:
import torch
import soundfile as sf
import librosa
from pathlib import Path

SAMPLING_RATE = 16000
INFER_BATCH = 1
INFER_BEAMS = 1

model.eval()

forced_ids = processor.get_decoder_prompt_ids(
    language="vi",
    task="translate"
)

# ⚠️ CHANGE THIS to your competition test folder
TEST_DIR = Path("/path/to/competition/test")
test_wavs = sorted(TEST_DIR.glob("*.wav"))

print("Num files:", len(test_wavs))


def load_audio(path):
    audio, sr = sf.read(str(path))

    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    if sr != SAMPLING_RATE:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=SAMPLING_RATE)

    return audio.astype("float32")


for i in range(0, len(test_wavs), INFER_BATCH):

    batch_files = test_wavs[i:i+INFER_BATCH]
    audios = [load_audio(p) for p in batch_files]

    inputs = processor.feature_extractor(
        audios,
        sampling_rate=SAMPLING_RATE,
        return_tensors="pt",
        padding=True
    )

    input_features = inputs.input_features.to(
        device,
        dtype=next(model.parameters()).dtype
    )

    with torch.no_grad():
        generated_ids = model.generate(
            input_features,
            forced_decoder_ids=forced_ids,
            num_beams=INFER_BEAMS,
        )

    texts = processor.tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )

    for p, t in zip(batch_files, texts):
        print("\n====================")
        print("clip_id:", p.stem)
        print("prediction:", t.strip())

Num files: 0


In [ ]:
import pandas as pd
from jiwer import wer
from sacrebleu.metrics import BLEU

bleu = BLEU()

# --- clean text ---
preds = [p.strip() for p in predictions]
refs  = [r.strip() for r in val_refs]

# --- compute metrics ---
bleu_score = bleu.corpus_score(preds, [refs]).score
wer_score = wer(refs, preds)

print("BLEU:", bleu_score)
print("WER:", wer_score)

# ==========================================================
# THIS is the table you need for error analysis
# ==========================================================

df = pd.DataFrame({
    "reference": refs,
    "prediction": preds,
})

df["ref_len"] = df["reference"].str.split().str.len()
df["pred_len"] = df["prediction"].str.split().str.len()
df["len_diff"] = df["pred_len"] - df["ref_len"]

# error flags (simple but very useful)
df["empty_pred"] = df["prediction"].str.strip().eq("")
df["short_pred"] = df["pred_len"] < 3
df["long_pred"] = df["pred_len"] > df["ref_len"] * 1.5

df.to_csv("error_analysis_table.csv", index=False)

df.head(n = 10)

BLEU: 10.700022156020545
WER: 0.8057346686788311


,reference,prediction,ref_len,pred_len,len_diff,empty_pred,short_pred,long_pred
0,some animals such as elephants and giraffes te...,Some animals like elephants and tigers often r...,20,21,1,False,False,False
1,a satellite phone is not generally a replaceme...,"In general, a cell phone is not a replacement ...",31,32,1,False,False,False
2,handicraft products might be defined as antiqu...,"The handmade products can be considered old, a...",15,20,5,False,False,False
3,then lakkha singh took the lead in singing the...,Then La Ka Sing holds the main role when singi...,10,14,4,False,False,False
4,some patients might have contracted the bug in...,Dr. Mon thinks that some patients may have bee...,21,23,2,False,False,False
5,i don't know if you realize it or not but most...,"I don't know if you can recognize it, but most...",22,22,0,False,False,False
6,argentina is well known for having one of the ...,Argentina is famous for being one of the best ...,17,16,-1,False,False,False
7,the final score was a one-point victory 21 to ...,The total score is a win with a gap of only 1....,18,13,-5,False,False,False
8,if you visit the arctic or antarctic areas in ...,If you visit the North or South Pole in the wi...,27,27,0,False,False,False
9,research in ai involves making machines to aut...,The research of artificial intelligence includ...,13,16,3,False,False,False


## 6. Predict on the competition test set

Merge the LoRA adapters back into the base weights and translate every test clip. (How the model decodes is set in the cell below - worth a close read.)

In [ ]:
def load_audio(path: Path) -> np.ndarray:
    """Read one .wav file as a mono, 16 kHz float array (what Whisper expects)."""
    array, sr = sf.read(str(path), dtype="float32", always_2d=False)
    if array.ndim > 1:
        array = array.mean(axis=1)          # stereo → mono
    if sr != SAMPLING_RATE:
        import librosa
        array = librosa.resample(array, orig_sr=sr, target_sr=SAMPLING_RATE)
    return array.astype(np.float32)

**Translate a batch of clips.** A helper that loads each `.wav`, runs the model, and decodes the output ids back into English text - `INFER_BATCH` clips at a time.

In [ ]:
def translate_wavs(model, wavs: List[Path]) -> List[str]:
    """Translate a list of .wav files into English strings, INFER_BATCH clips at a time."""
    model.eval()
    forced_ids = processor.get_decoder_prompt_ids(language="vi", task="translate")
    model_dtype = next(model.parameters()).dtype
    outputs: List[str] = []
    for start in range(0, len(wavs), INFER_BATCH):
        batch = wavs[start:start + INFER_BATCH]
        arrays = [load_audio(p) for p in batch]
        inputs = processor.feature_extractor(
            arrays, sampling_rate=SAMPLING_RATE, return_tensors="pt", padding=True
        )
        feats = inputs.input_features.to(device, dtype=model_dtype)
        attn  = inputs.attention_mask.to(device) if "attention_mask" in inputs else None
        with torch.no_grad():
            ids = model.generate(
                feats,
                attention_mask=attn,
                forced_decoder_ids=forced_ids,
                num_beams=8,
                no_repeat_ngram_size=2,   # block repeated bigrams
                # max_new_tokens=225,
            )
        outputs.extend(processor.tokenizer.batch_decode(ids, skip_special_tokens=True))
        if (start // INFER_BATCH) % 20 == 0:
            print(f"  …{start + len(batch)}/{len(wavs)}")
    return outputs

**Run it on all the test clips.** This is the slow step - progress prints every ~160 clips.

In [ ]:
# Fold the LoRA adapter into the base weights, then translate every competition clip.
ft_model = model.merge_and_unload().to(device)
print(f"Translating {len(TEST_WAVS)} test clips with beam={INFER_BEAMS}…")
predictions = translate_wavs(ft_model, TEST_WAVS)
print("Done.")

## 7. Write `submission.csv`

Two columns - `clip_id`, `prediction` - one row per test clip. This is the file Kaggle will score.

In [ ]:
# Build the submission: one row per clip, columns clip_id + prediction.
# Any empty prediction is replaced with "_" so every clip has a non-empty entry.
submission = pd.DataFrame({
    "clip_id":    [p.stem for p in TEST_WAVS],
    "prediction": [p.strip() if p.strip() else "_" for p in predictions],
}).sort_values("clip_id").reset_index(drop=True)

submission.to_csv(SUBMISSION, index=False)
print(f"Wrote {len(submission)} predictions to {SUBMISSION}")
print(submission.head())

In [ ]:
import shutil

# Define the path in your Google Drive where you want to save the submission file
gdrive_submission_path = '/content/drive/MyDrive/submission.csv'

# Copy the submission file to Google Drive
shutil.copy(SUBMISSION, gdrive_submission_path)
print(f"Copied submission.csv to {gdrive_submission_path}")

### Generate Submissions for Multiple Checkpoints

This section will iterate through the specified checkpoint directories, load each fine-tuned model and its processor, generate predictions on the test set, and save a unique `submission.csv` for each checkpoint. This allows you to evaluate the performance of different checkpoints.

In [ ]:
import sys
import subprocess

print("Installing sentencepiece...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentencepiece"])
print("sentencepiece installed.")

In [ ]:
import os
from pathlib import Path
from peft import PeftModel
from transformers import WhisperForConditionalGeneration, WhisperProcessor
import pandas as pd
import numpy as np
import soundfile as sf
import torch
import librosa
import sys
import subprocess
import kagglehub
import gc # Import garbage collector

# Set environment variable to potentially mitigate CUDA out of memory issues
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# --- Environment Setup (Crucial for independence) ---
print("-- Setting up environment for submission generation --")

# Install sentencepiece (if not already present and causing issues)
print("Installing sentencepiece (if not already present)...")
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentencepiece"])
    print("sentencepiece installed successfully.")
except Exception as e:
    print(f"Warning: Could not install sentencepiece. Ensure it's available. Error: {e}")

# Ensure torchao is installed and updated to a compatible version
print("Ensuring compatible torchao version...")
try:
    # Uninstall any existing torchao to force a fresh install
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torchao>=0.16.0"])
    print("torchao updated successfully.")
except Exception as e:
    print(f"Error ensuring compatible torchao version: {e}. Please ensure torchao can be installed/upgraded.")

# --- Reinstall transformers and peft to resolve potential version conflicts ---
print("Attempting to reinstall transformers and peft...")
try:
    # Uninstall existing versions
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "peft"])
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "transformers"])

    # Install transformers (let it handle its dependencies)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers"])
    # Install peft (let it handle its dependencies, compatible with transformers)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "peft"])
    print("transformers and peft reinstalled successfully.")
except Exception as e:
    print(f"Error reinstalling transformers or peft: {e}. Please check your environment.")

# Log into KaggleHub and download competition data (if not already done).
# This assumes Kaggle API credentials are set up (e.g., via Colab secrets or kaggle.json).
print("Attempting to log into KaggleHub and download competition data...")
assignment_2_speech_translation_path = None
try:
    kagglehub.login()
    assignment_2_speech_translation_path = kagglehub.competition_download('assignment-2-speech-translation')
    print(f"Kaggle data downloaded to: {assignment_2_speech_translation_path}")
except Exception as e:
    print(f"Error logging into KaggleHub or downloading data: {e}. Please ensure Kaggle API credentials are properly configured (KAGGLE_USERNAME, KAGGLE_KEY in Colab secrets or kaggle.json file). ")

if assignment_2_speech_translation_path is None:
    raise ValueError("Kaggle competition data path is not set. Cannot proceed without the test data. Please resolve Kaggle download issues.")

# --- Global Parameters (Knobs from the original notebook, re-defined for independence) ---
MODEL_ID         = "openai/whisper-medium" # Changed from "openai/whisper-medium" to match training model
INFER_BEAMS      = 2                         # Further reduced beams to save memory
SAMPLING_RATE    = 16_000                    # Whisper expects 16 kHz audio
INFER_BATCH      = 2                         # Further reduced batch size to save memory
device           = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {device}")
print(f"Model ID for base model: {MODEL_ID}")
print(f"Inference beams: {INFER_BEAMS}")
print(f"Sampling rate: {SAMPLING_RATE}")
print(f"Inference batch size: {INFER_BATCH}")

# --- Helper Function: Find Test Directory ---
def find_test_dir(comp_data_path: str) -> Path:
    """Find the folder of competition test .wav files."""
    comp_root = Path(comp_data_path)
    candidates = []
    if comp_root.exists():
        candidates = sorted(comp_root.glob("*/test"))

    if not candidates:
        raise FileNotFoundError(
            "No 'test/' directory found under the specified competition data path. "
            "Ensure the dataset was downloaded correctly and check its internal structure."
        )
    return max(candidates, key=lambda p: len(list(p.glob("*.wav"))))

TEST_DIR  = find_test_dir(assignment_2_speech_translation_path)
TEST_WAVS = sorted(TEST_DIR.glob("*.wav"))
print(f"Test directory found: {TEST_DIR}")
print(f"Found {len(TEST_WAVS)} test WAV files.")

# --- Helper Function: Load Audio ---
def load_audio(path: Path) -> np.ndarray:
    """Read one .wav file as a mono, 16 kHz float array (what Whisper expects)."""
    array, sr = sf.read(str(path), dtype="float32", always_2d=False)
    if array.ndim > 1:
        array = array.mean(axis=1)          # stereo → mono
    if sr != SAMPLING_RATE:
        # Ensure librosa is installed if needed for resampling
        try:
            import librosa
            array = librosa.resample(array, orig_sr=sr, target_sr=SAMPLING_RATE)
        except ImportError:
            print("Warning: librosa not found. Resampling skipped. Please install 'librosa' if audio files have different sample rates.")
            # If librosa is not available, proceed without resampling, which might cause issues
    return array.astype(np.float32)

# --- Helper Function: Translate WAVs ---
def translate_wavs(model_to_use, processor_to_use, wavs: list[Path]) -> list[str]:
    """Translate a list of .wav files into English strings, INFER_BATCH clips at a time."""
    model_to_use.eval()
    forced_ids = processor_to_use.get_decoder_prompt_ids(language="vi", task="translate")
    model_dtype = next(model_to_use.parameters()).dtype
    outputs: list[str] = []
    for start in range(0, len(wavs), INFER_BATCH):
        batch = wavs[start:start + INFER_BATCH]
        arrays = [load_audio(p) for p in batch]
        inputs = processor_to_use.feature_extractor(
            arrays, sampling_rate=SAMPLING_RATE, return_tensors="pt", padding=True
        )
        feats = inputs.input_features.to(device, dtype=model_dtype)
        attn  = inputs.attention_mask.to(device) if "attention_mask" in inputs else None
        with torch.no_grad():
            ids = model_to_use.generate(
                feats,
                attention_mask=attn,
                forced_decoder_ids=forced_ids,
                num_beams=INFER_BEAMS,
                no_repeat_ngram_size=2,
            )
        outputs.extend(processor_to_use.tokenizer.batch_decode(ids, skip_special_tokens=True))
        if (start // INFER_BATCH) % 20 == 0:
            print(f"  ...Processed {start + len(batch)}/{len(wavs)} files")
    return outputs

# --- Main Logic: Generate Submissions for Checkpoints ---
base_checkpoint_dirs = [
    '/content/drive/MyDrive/'
]

checkpoint_paths_to_process = []
for base_dir in base_checkpoint_dirs:
    base_dir_path = Path(base_dir)
    if base_dir_path.exists() and base_dir_path.is_dir():
        for item in base_dir_path.iterdir():
            if item.is_dir() and item.name.startswith('checkpoints_'):
                checkpoint_paths_to_process.append(str(item))
    else:
        print(f"Warning: Base directory '{base_dir}' not found. Skipping any checkpoints within it.")

if not checkpoint_paths_to_process:
    raise ValueError("No checkpoint directories found. Please ensure they exist and are named 'checkpoint-X'.")

print(f"Found {len(checkpoint_paths_to_process)} checkpoints to process: {checkpoint_paths_to_process}")

common_processor = WhisperProcessor.from_pretrained(MODEL_ID, language="vi", task="translate")

all_submissions = {}

for checkpoint_path in checkpoint_paths_to_process:
    print(f"\n--- Processing checkpoint: {checkpoint_path} ---")

    # Initialize variables to None before the try block
    base_model = None
    peft_model = None
    ft_model = None
    predictions = None
    submission_df = None

    try:
        # Load the base model
        base_model = WhisperForConditionalGeneration.from_pretrained(
            MODEL_ID, torch_dtype=torch.bfloat16, attn_implementation="sdpa"
        )

        # Load the PEFT (LoRA) model from the checkpoint and merge it
        peft_model = PeftModel.from_pretrained(base_model, checkpoint_path)
        ft_model = peft_model.merge_and_unload().to(device)

        # Set generation config for this model instance
        ft_model.generation_config.language = "vi"
        ft_model.generation_config.task = "translate"
        ft_model.generation_config.num_beams = INFER_BEAMS
        ft_model.generation_config.no_repeat_ngram_size = 2
        ft_model.generation_config.suppress_tokens = []
        ft_model.generation_config.forced_decoder_ids = None
        ft_model.config.suppress_tokens = None
        ft_model.config.begin_suppress_tokens = None
        ft_model.config.forced_decoder_ids = None

        # Generate predictions using the common_processor
        print("Generating predictions...")
        predictions = translate_wavs(ft_model, common_processor, TEST_WAVS)
        print("Finished generating predictions.")

        # Create submission DataFrame
        submission_df = pd.DataFrame({
            "clip_id": [p.stem for p in TEST_WAVS],
            "prediction": [p.strip() if p.strip() else "_" for p in predictions],
        }).sort_values("clip_id").reset_index(drop=True)

        # Define submission output path for this checkpoint
        checkpoint_name = Path(checkpoint_path).name
        submission_output_dir = Path(checkpoint_path) / "submissions"
        submission_output_dir.mkdir(parents=True, exist_ok=True)
        submission_file_path = submission_output_dir / f"submission_{checkpoint_name}.csv"

        submission_df.to_csv(submission_file_path, index=False)
        print(f"Wrote {len(submission_df)} predictions to {submission_file_path}")
        print(submission_df.head())

        all_submissions[checkpoint_name] = submission_df

    except Exception as e:
        print(f"Error processing checkpoint {checkpoint_path}: {e}")

    finally:
        # Clear memory after each checkpoint processing attempt
        # Only delete if they were actually assigned (i.e., not None)
        if base_model is not None: del base_model
        if peft_model is not None: del peft_model
        if ft_model is not None: del ft_model
        if predictions is not None: del predictions
        if submission_df is not None: del submission_df

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
        print(f"Memory cleared for checkpoint {checkpoint_path}")

print("\n--- All submission files generated. ---")

-- Setting up environment for submission generation --
Installing sentencepiece (if not already present)...
sentencepiece installed successfully.
Ensuring compatible torchao version...
torchao updated successfully.
Attempting to reinstall transformers and peft...
transformers and peft reinstalled successfully.
Attempting to log into KaggleHub and download competition data...


Kaggle data downloaded to: /root/.cache/kagglehub/competitions/assignment-2-speech-translation
Using device: cuda
Model ID for base model: openai/whisper-medium
Inference beams: 2
Sampling rate: 16000
Inference batch size: 2
Test directory found: /root/.cache/kagglehub/competitions/assignment-2-speech-translation/public_release_v2/test
Found 2481 test WAV files.
Found 2 checkpoints to process: ['/content/drive/MyDrive/checkpoints_best', '/content/drive/MyDrive/checkpoints_sbt']


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.99k [00:00<?, ?B/s]

...

Generating predictions...


[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transform

  ...Processed 2/2481 files
  ...Processed 42/2481 files
  ...Processed 82/2481 files
  ...Processed 122/2481 files
  ...Processed 162/2481 files
  ...Processed 202/2481 files
  ...Processed 242/2481 files
  ...Processed 282/2481 files
  ...Processed 322/2481 files
  ...Processed 362/2481 files
  ...Processed 402/2481 files
  ...Processed 442/2481 files
  ...Processed 482/2481 files
  ...Processed 522/2481 files
  ...Processed 562/2481 files
  ...Processed 602/2481 files
  ...Processed 642/2481 files
  ...Processed 682/2481 files
  ...Processed 722/2481 files
  ...Processed 762/2481 files
  ...Processed 802/2481 files
  ...Processed 842/2481 files
  ...Processed 882/2481 files
  ...Processed 922/2481 files
  ...Processed 962/2481 files
  ...Processed 1002/2481 files
  ...Processed 1042/2481 files
  ...Processed 1082/2481 files
  ...Processed 1122/2481 files
  ...Processed 1162/2481 files
  ...Processed 1202/2481 files
  ...Processed 1242/2481 files
  ...Processed 1282/2481 files
  ...P

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

Generating predictions...
  ...Processed 2/2481 files
  ...Processed 42/2481 files
  ...Processed 82/2481 files
  ...Processed 122/2481 files
  ...Processed 162/2481 files
  ...Processed 202/2481 files


## Fixes:
### Decoding:
- Try INFER_BEAMS = 1, 2, 4, 6, 8
    + Beam=1  BLEU=8.36
    + Beam=2  BLEU=9.24
    + Beam=4  BLEU=9.98
    + Beam=8  BLEU=10.77

- Try no_repeat_ngram_size=0
    + Beam=1  BLEU=5.41
    + Beam=2  BLEU=4.29
    + Beam=4  BLEU=4.23
    + Beam=8  BLEU=2.86

### Fix Training Hyperparameters:
- Try lr = 3e-4
- Try lr = 1e-4
- Try r=16, alpha=32
- Try r=32, alpha=64
- Try epoch=5, epoch=8
- Try Whisper-Medium or whisper-large-v3

### Find bug
- Try padding=True instead of max_length

## You have a baseline. Now beat it.

Everything above runs, but the defaults are plain and *at least one line is not doing what its comment claims.* This unmodified run will land a **modest** BLEU score. Climbing from here is the whole assignment - and it rewards understanding the code, not copying it.

From here there are **no instructions, only questions.** Pick directions based on what your own error analysis tells you; you won't have time for all of them.

### Rule 0 - measure before you change anything

You get **5 submissions a day**, so the leaderboard is a terrible debugging tool. Build a **local scorer** instead. You already have a held-out split (`dataset["validation"]`) with English references - translate it and score with the same metric the competition uses:

    import sacrebleu
    bleu = sacrebleu.corpus_bleu(predictions, [references])
    print(bleu.score)

Now you can compare two ideas in minutes. **Change one thing, re-score, write the number down.**

Two things worth sitting with:
- **Domain.** Look at the test filenames - what kind of audio is this (studio-read sentences? something messier, chunked out of longer recordings)? How close is your validation data to it? A validation set from the wrong domain can tell you a confident lie.
- **The metric.** BLEU is picky about casing and punctuation. If a change "improves" your score by only altering how text is normalized, did the *translation* get better - or just the *scoring*?

### Rule 1 - look at what's failing

Pull 10–20 clips your model gets wrong and actually read (or listen to) them:
- Is it **mishearing the Vietnamese** (a speech problem) or **mistranslating** what it heard (a language problem)? Different fixes.
- Does it **hallucinate** text that isn't there, **repeat** itself, or **drop** numbers and names?
- Are the failures concentrated - long clips? noisy clips? one speaker or topic?

Your answers should decide which direction below you try first.

### Directions to explore

Each is a **hypothesis to test**, not a recipe. For every one: *why might it help?* → *how would you try it?* → *what number tells you it worked?*

- **Decoding (cheapest - start here, no training).** The decoding settings in this notebook are choices, and choices can be wrong. What does each generation argument actually *do*? What happens to BLEU and to runtime if you change or remove them? *This is where the cheapest gains - and a deliberately bad default - are hiding.*
- **The model.** The rules allow **any Whisper variant.** Bigger models are more accurate but slower and hungrier for memory. Where's the sweet spot for your GPU budget? *Measure BLEU against wall-clock and memory, not BLEU alone.*
- **Training data.** What other **public** Vietnamese-audio → English-text data exists? Does *more* data win, or does **domain match** (audio that looks like the test clips) matter more? What's the risk of training on machine-translated or otherwise noisy labels?
- **How you train.** `NUM_EPOCHS`, the LoRA rank/α, the learning rate - starting points, not answers. How long before the model **overfits** your (small) data, and how would you *see* it? *Watch validation BLEU per epoch, not training loss.*
- **Audio preprocessing.** Real recordings have silence, noise, uneven loudness, and some clips may be longer than the model's input window. Does **cleaning the audio first** (trimming silence, normalizing loudness, splitting long clips) help? Start with the failure cases from Rule 1.
- **Combining models (advanced).** Could several models - or several decodings of one model - be combined into a better answer? What does it cost at inference, and when is the gain worth it?

### Working like a researcher

- **One change at a time** - change three things and a moving score teaches you nothing about which one mattered.
- **Validation first, submission second** - spend your 5 daily submissions confirming what already looked good locally.
- **Keep a log:** config → validation BLEU → leaderboard. Patterns show up fast.
- **Watch for divergence:** validation BLEU climbing while the leaderboard doesn't is itself a clue (overfitting? a domain gap?).

There is no single intended solution. The points come from looking closely at the data and testing your own ideas. **Submit early, iterate often.**